<a href="https://colab.research.google.com/github/gredy/2021Z-DataVisualizationTechniques/blob/master/2_Layer_MLP_in_(Numpy).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Implement a 2‑layer MLP entirely in NumPy:

Load and preprocess Fashion‑MNIST (flatten & normalize).
Define forward pass (linear → ReLU → linear → softmax).
Implement backpropagation and SGD updates.
Train on 5,000 samples; plot train/val loss & accuracy.
Experiment with learning rate and depth.

In [ ]:
# implement a 2 layer MLP entirely in Numpy


In [ ]:
import numpy as np
from tensorflow.keras.datasets import fashion_mnist
import matplotlib.pyplot as plt

# Load Fashion-MNIST dataset
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

# Preprocess data
# Flatten images
x_train_flat = x_train.reshape(x_train.shape[0], -1)
x_test_flat = x_test.reshape(x_test.shape[0], -1)

# Normalize pixel values to [0, 1]
x_train_norm = x_train_flat.astype('float32') / 255.0
x_test_norm = x_test_flat.astype('float32') / 255.0

# One-hot encode labels
def one_hot_encode(labels, num_classes):
    one_hot = np.zeros((labels.shape[0], num_classes))
    one_hot[np.arange(labels.shape[0]), labels] = 1
    return one_hot

num_classes = 10
y_train_one_hot = one_hot_encode(y_train, num_classes)
y_test_one_hot = one_hot_encode(y_test, num_classes)

print(f"x_train_norm shape: {x_train_norm.shape}")
print(f"y_train_one_hot shape: {y_train_one_hot.shape}")
print(f"x_test_norm shape: {x_test_norm.shape}")
print(f"y_test_one_hot shape: {y_test_one_hot.shape}")

Next, we'll define helper functions for activation functions (ReLU, Softmax) and their derivatives, and initialize the weights and biases for our 2-layer MLP.

In [ ]:
def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True)) # Subtract max for numerical stability
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

def initialize_parameters(input_size, hidden_size, output_size):
    # He initialization for ReLU
    W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2. / input_size)
    b1 = np.zeros((1, hidden_size))

    # Xavier initialization for output layer (linear activation followed by softmax)
    W2 = np.random.randn(hidden_size, output_size) * np.sqrt(2. / (hidden_size + output_size))
    b2 = np.zeros((1, output_size))

    return {'W1': W1, 'b1': b1, 'W2': W2, 'b2': b2}

# Define network architecture
input_size = x_train_norm.shape[1]  # 28 * 28 = 784
hidden_size = 128 # Example hidden layer size
output_size = num_classes # 10 classes for Fashion-MNIST

params = initialize_parameters(input_size, hidden_size, output_size)

print("Parameters initialized:")
for key, value in params.items():
    print(f"{key} shape: {value.shape}")

Now we'll implement the forward pass, which calculates the output of the network given an input.

In [ ]:
def forward_pass(X, params):
    W1, b1, W2, b2 = params['W1'], params['b1'], params['W2'], params['b2']

    # Layer 1 (Hidden Layer)
    Z1 = np.dot(X, W1) + b1
    A1 = relu(Z1)

    # Layer 2 (Output Layer)
    Z2 = np.dot(A1, W2) + b2
    A2 = softmax(Z2)

    return Z1, A1, Z2, A2

# Example forward pass for a small batch
batch_size = 64
X_sample = x_train_norm[:batch_size]

Z1_sample, A1_sample, Z2_sample, A2_sample = forward_pass(X_sample, params)

print(f"Output of forward pass (A2_sample) shape: {A2_sample.shape}")

Next, we'll implement the loss function (cross-entropy) and the backpropagation algorithm to calculate gradients for updating our weights and biases.

In [ ]:
def cross_entropy_loss(y_pred, y_true):
    # Avoid log(0) by clipping predictions
    epsilon = 1e-12
    y_pred = np.clip(y_pred, epsilon, 1. - epsilon)
    loss = -np.sum(y_true * np.log(y_pred)) / y_true.shape[0]
    return loss

def backward_pass(X, Y_true, params, Z1, A1, Z2, A2):
    W1, b1, W2, b2 = params['W1'], params['b1'], params['W2'], params['b2']
    m = X.shape[0] # Number of samples in the batch

    # Output layer gradients
    dZ2 = A2 - Y_true  # Derivative of cross-entropy with softmax
    dW2 = np.dot(A1.T, dZ2) / m
    db2 = np.sum(dZ2, axis=0, keepdims=True) / m

    # Hidden layer gradients
    dA1 = np.dot(dZ2, W2.T)
    dZ1 = dA1 * relu_derivative(Z1)
    dW1 = np.dot(X.T, dZ1) / m
    db1 = np.sum(dZ1, axis=0, keepdims=True) / m

    grads = {'dW1': dW1, 'db1': db1, 'dW2': dW2, 'db2': db2}
    return grads

# Test backprop with sample data
Y_sample = y_train_one_hot[:batch_size]
grads_sample = backward_pass(X_sample, Y_sample, params, Z1_sample, A1_sample, Z2_sample, A2_sample)

print("Gradients calculated:")
for key, value in grads_sample.items():
    print(f"{key} shape: {value.shape}")

Finally, we'll implement the training loop using Stochastic Gradient Descent (SGD), train the model on 5,000 samples, and plot the training and validation loss and accuracy.

In [ ]:
def update_parameters(params, grads, learning_rate):
    params['W1'] -= learning_rate * grads['dW1']
    params['b1'] -= learning_rate * grads['db1']
    params['W2'] -= learning_rate * grads['dW2']
    params['b2'] -= learning_rate * grads['db2']
    return params

def calculate_accuracy(y_pred, y_true):
    predictions = np.argmax(y_pred, axis=1)
    true_labels = np.argmax(y_true, axis=1)
    return np.mean(predictions == true_labels)

# Training parameters
learning_rate = 0.01 # Experiment with this value
epochs = 50
batch_size = 64
train_samples = 5000 # Train on 5,000 samples

# Use a subset of the data for training and validation
X_train_subset = x_train_norm[:train_samples]
Y_train_subset = y_train_one_hot[:train_samples]

# Split into training and validation sets (80/20 split for example)
split_idx = int(train_samples * 0.8)
X_train_actual = X_train_subset[:split_idx]
Y_train_actual = Y_train_subset[:split_idx]
X_val = X_train_subset[split_idx:]
Y_val = Y_train_subset[split_idx:]

# Re-initialize parameters for fresh training
params = initialize_parameters(input_size, hidden_size, output_size)

history = {'train_loss': [], 'val_loss': [], 'train_accuracy': [], 'val_accuracy': []}

print("Starting training...")
for epoch in range(epochs):
    # Shuffle training data
    permutation = np.random.permutation(X_train_actual.shape[0])
    X_shuffled = X_train_actual[permutation]
    Y_shuffled = Y_train_actual[permutation]

    epoch_train_loss = 0
    epoch_train_acc = 0
    num_batches = 0

    for i in range(0, X_shuffled.shape[0], batch_size):
        X_batch = X_shuffled[i:i+batch_size]
        Y_batch = Y_shuffled[i:i+batch_size]

        # Forward pass
        Z1, A1, Z2, A2 = forward_pass(X_batch, params)

        # Calculate loss and accuracy for batch
        loss = cross_entropy_loss(A2, Y_batch)
        accuracy = calculate_accuracy(A2, Y_batch)

        epoch_train_loss += loss
        epoch_train_acc += accuracy
        num_batches += 1

        # Backward pass
        grads = backward_pass(X_batch, Y_batch, params, Z1, A1, Z2, A2)

        # Update parameters
        params = update_parameters(params, grads, learning_rate)

    # Average training metrics for the epoch
    avg_train_loss = epoch_train_loss / num_batches
    avg_train_acc = epoch_train_acc / num_batches
    history['train_loss'].append(avg_train_loss)
    history['train_accuracy'].append(avg_train_acc)

    # Validation pass
    _, _, _, A2_val = forward_pass(X_val, params)
    val_loss = cross_entropy_loss(A2_val, Y_val)
    val_accuracy = calculate_accuracy(A2_val, Y_val)
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_accuracy)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{epochs} - Train Loss: {avg_train_loss:.4f}, Train Acc: {avg_train_acc:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.4f}")

print("Training complete!")

# Plotting results
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history['train_accuracy'], label='Train Accuracy')
plt.plot(history['val_accuracy'], label='Val Accuracy')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

Feel free to experiment with the `learning_rate` and `hidden_size` to see how they impact the model's performance.